# Did our preprocessing drop information that separates the overlapping classes

About 80% of Analysis and Backdoor flows have an identical-feature twin of another class among the features we kept. This notebook returns to the raw first file (UNSW-NB15_1.csv, the paper's data) and asks whether the columns we dropped (state, loss counts, window, TCP sequence numbers, round-trip time, flag columns, source port) separate those flows. Duplicate rows are removed on all columns, which gives the same 637,891 flows as the preprocessed data. Source and destination IPs and the time stamps are not used.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

RAW_DIR = "/kaggle/input/datasets/shahiismyname/unsw-nb15-raw-dataset"
names = pd.read_csv(f"{RAW_DIR}/NUSW-NB15_features.csv", encoding="latin-1")["Name"].str.replace(" ", "").str.strip().tolist()
d = pd.read_csv(f"{RAW_DIR}/UNSW-NB15_RAW_FILES/UNSW-NB15_1.csv", header=None, names=names, dtype={"sport": str, "dsport": str}, low_memory=False)
d["attack_cat"] = d["attack_cat"].fillna("Normal").str.strip().replace({"Backdoors": "Backdoor"})
for c in ["sport", "dsport"]:
    d[c] = pd.to_numeric(d[c], errors="coerce").fillna(-1)
for c in ["proto", "service", "state"]:
    d[c] = d[c].astype("category").cat.codes

kept = ["dsport", "dur", "sbytes", "dbytes", "sttl", "dttl", "Sload", "Spkts", "Dpkts", "smeansz", "trans_depth", "res_bdy_len", "Sjit", "Djit",
        "Sintpkt", "Dintpkt", "synack", "ackdat", "ct_state_ttl", "ct_flw_http_mthd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm",
        "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm", "Dload", "swin", "dmeansz", "proto", "service"]
dropped = ["state", "sloss", "dloss", "dwin", "stcpb", "dtcpb", "tcprtt", "is_sm_ips_ports", "is_ftp_login", "ct_ftp_cmd", "sport"]
sets = {"features we kept": kept, "kept + dropped columns": kept + dropped,
        "kept + dropped, without TCP sequence numbers": kept + [c for c in dropped if c not in ("stcpb", "dtcpb")]}

d = d.drop_duplicates(subset=kept + dropped + ["attack_cat"]).reset_index(drop=True)
classes = sorted(d["attack_cat"].unique())
y = d["attack_cat"].map({c: i for i, c in enumerate(classes)}).to_numpy()
print(len(d), "flows after duplicate removal on all columns")

## 1. Identical-feature twins of another class (%)

In [ ]:
twins = {}
for name, cols in sets.items():
    g = pd.DataFrame({"k": pd.util.hash_pandas_object(d[cols], index=False).to_numpy(), "y": y})
    flag = g.groupby("k")["y"].transform("nunique") > 1
    twins[name] = (g.assign(t=flag).groupby("y")["t"].mean() * 100).round(1).rename(lambda i: classes[i])
twins = pd.DataFrame(twins)
print(twins.to_string())
twins.to_csv("/kaggle/working/dropped_features_twins.csv")

## 2. LightGBM with each column set

Class-weighted LightGBM, stratified 5:2:3 split, early stopping on validation log loss, one seed.

In [ ]:
tr, rest = train_test_split(np.arange(len(d)), train_size=0.5, stratify=y, random_state=0)
va, te = train_test_split(rest, train_size=0.4, stratify=y[rest], random_state=0)
res = {}
for name, cols in sets.items():
    m = lgb.LGBMClassifier(class_weight="balanced", learning_rate=0.08, num_leaves=31, n_estimators=400, random_state=0, verbose=-1, n_jobs=4)
    m.fit(d.loc[tr, cols], y[tr], eval_set=[(d.loc[va, cols], y[va])], callbacks=[lgb.early_stopping(20, verbose=False)])
    p = m.predict(d.loc[te, cols])
    f = f1_score(y[te], p, labels=range(len(classes)), average=None, zero_division=0)
    res[name] = {**dict(zip(classes, f.round(3))), "macro F1": round(f.mean(), 3), "weighted F1": round(f1_score(y[te], p, average="weighted"), 4), "trees": m.best_iteration_}
res = pd.DataFrame(res).T
print(res.to_string())
res.to_csv("/kaggle/working/dropped_features_lightgbm.csv")